# BERT XAI

This notebook reproduces the experiments using the refactored package modules instead of inline code.

In [1]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Using src path:", SRC)

Project root: /users/eleves-a/2022/vincent.zhou/Documents/XAI/XAI
Using src path: /users/eleves-a/2022/vincent.zhou/Documents/XAI/XAI/src


In [2]:
from xai_bert import (
    BertHiddenStateRunner,
    anisotropy_all_layers_before_after,
    compare_outlier_stats_before_after_clipping,
    compute_mean_vectors_over_corpus,
    compute_outlier_dimension_stats_over_corpus,
    flatten_valid_tokens,
    load_sst2_text_splits,
    load_wic_splits,
    load_wikitext_103_texts,
    context_words_for_text,
    plot_anisotropy_before_after,
    plot_mean_vectors,
    print_anisotropy_results,
    print_before_after_comparison,
    print_outlier_stats,
    run_probe_for_selected_layers_with_splits,
    select_primary_outlier_dims,
    top_k_dims,
    compute_word_self_similarity_selected_layers,
    detect_token_outlier_responsibility,
    print_token_outlier_summary,
    evaluate_wic_with_threshold,
    pretrain_tiny_bert_mlm,
    evaluate_mlm_loss,
)

In [3]:
runner = BertHiddenStateRunner(model_name="bert-base-uncased", max_length=64)

texts = [
    "The cat is sleeping on the sofa.",
    "A transformer model produces contextualized embeddings.",
    "This paper studies positional artifacts in BERT.",
    "Layer normalization can amplify certain dimensions.",
    "The quick brown fox jumps over the lazy dog.",
] * 200

print("num texts:", len(texts))
print("num unique texts:", len(set(texts)))
print("runner model:", "bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


num texts: 1000
num unique texts: 5
runner model: bert-base-uncased


In [4]:
train_texts, val_texts, test_texts = load_sst2_text_splits(
    max_train=5000,
    seed=42,
)

print("SST-2 split sizes:")
print("  train:", len(train_texts))
print("  val  :", len(val_texts))
print("  test :", len(test_texts))

SST-2 split sizes:
  train: 5000
  val  : 872
  test : 872


## Experiment 1: Mean Vectors Per Layer

In [ ]:
mean_vectors = compute_mean_vectors_over_corpus(
    runner=runner,
    texts=train_texts,
    batch_size=16,
    exclude_embedding_layer=True,
    remove_special_tokens=True,
)

plot_mean_vectors(mean_vectors)

## Experiment 2: Outlier Dimensions

In [ ]:
outlier_stats = compute_outlier_dimension_stats_over_corpus(
    runner=runner,
    texts=train_texts,
    batch_size=16,
    exclude_embedding_layer=True,
    remove_special_tokens=True,
)

print_outlier_stats(outlier_stats)

layer_3_min_top10 = top_k_dims(outlier_stats[3]["min_counter"], k=10)
print("Top-10 min dims for layer 3:", layer_3_min_top10)

## Experiment 3: Single-Layer Clipping Check

In [ ]:
batch_out = runner.run_batch(texts[:16])

flat_l1 = flatten_valid_tokens(
    batch_output=batch_out,
    layer_idx=1,
    remove_special_tokens=False,
)

dims_to_clip = select_primary_outlier_dims(outlier_stats[1], mode="min")
comparison = compare_outlier_stats_before_after_clipping(
    flat_layer=flat_l1,
    dims_to_clip=dims_to_clip,
)

print_before_after_comparison(comparison)

## Experiment 4: Anisotropy Before/After Clipping

In [ ]:
anisotropy_results = anisotropy_all_layers_before_after(
    runner=runner,
    texts=texts,
    outlier_stats=outlier_stats,
    batch_size=16,
    remove_special_tokens=False,
    exclude_embedding_layer=True,
    clip_mode="min",
    n_pairs=1000,
    seed=42,
)

print_anisotropy_results(anisotropy_results)
plot_anisotropy_before_after(anisotropy_results)

## Experiment 5: Position Probe on SST-2 (GLUE)

In [ ]:
layer_probe_results = run_probe_for_selected_layers_with_splits(
    runner=runner,
    train_texts=train_texts,
    val_texts=val_texts,
    test_texts=test_texts,
    layers=(1, 3, 6, 9, 12),
    max_position=300,
    build_batch_size=16,
    probe_batch_size=128,
    num_epochs=5,
    lr=1e-3,
    remove_special_tokens=True,
)

## Experiment 6: Self similarity

Given a word $w$ and $n$ different sentences $s_1, s_2, \dots, s_n$ which contain such word, $f_l^i(w)$ is the internal representation of $w$ in sentence $s_i$ in the $l^{th}$ layer. The average self-similarity of $w$ in the $l^{th}$ layer is then defined as:

\begin{equation}
\text{SelfSim}_l(w) = \frac{\sum_{i=1}^n \sum_{j=i+1}^n \cos\left(f_l^i(w), f_l^j(w)\right)}{n(n - 1)}
\end{equation}

In [ ]:
train_texts_total, _, _ = load_sst2_text_splits(max_train=None, max_val=0, max_test=0, seed=42)
selfsim_target_words = context_words_for_text(train_texts_total, n_word=10, n_sentence_min=10, random_select=True)

In [ ]:
print("Selected target words for self-similarity analysis:", selfsim_target_words)

In [ ]:
layers = (1, 3, 6, 9, 12)

In [ ]:
dims_to_clip_by_layer = {
    layer_idx: stats["top_min_dim"]
    for layer_idx, stats in outlier_stats.items()
}
self_similarity_results = []
for word in selfsim_target_words:
	sentences_with_word = [
		text for text in train_texts_total
		if word in text.split()
	]

	self_similarity_results.append(
		compute_word_self_similarity_selected_layers(
			runner=runner,
			texts=sentences_with_word,
			dims_to_clip=dims_to_clip_by_layer,
		layer_indices=layers,
		anisotropy=anisotropy_results,
		)
	)

In [ ]:
self_similarity_results_abs = []
for i in range(len(self_similarity_results)):
    tmp = {}
    for layer in layers:
        tmp[layer] = {
			"before": abs(self_similarity_results[i][layer]["before"]),
			"after": abs(self_similarity_results[i][layer]["after"]),
		}
    self_similarity_results_abs.append(tmp)

In [ ]:
results = {"before": {layer:0 for layer in layers}, "after": {layer:0 for layer in layers}}
for i in range(len(self_similarity_results)):
    for layer in layers:
        results["before"][layer] += self_similarity_results_abs[i][layer]["before"]
        results["after"][layer] += self_similarity_results_abs[i][layer]["after"]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(layers, [results["before"][layer] for layer in layers], marker="o", label="Before Clipping")
plt.plot(layers, [results["after"][layer] for layer in layers], marker="o", label="After Clipping")
plt.xlabel("Layer")
plt.ylabel("Average Self-Similarity")
plt.title("Average Self-Similarity of Target Words Before and After Clipping")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import math

n_words = len(selfsim_target_words)
n_cols = 3
n_rows = max(1, math.ceil(n_words / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.5 * n_cols, 3.2 * n_rows),
    sharex=True,
    sharey=True,
    constrained_layout=True,
 )

# Normalize axes into a flat list for consistent indexing
if n_rows == 1 and n_cols == 1:
    axes = [axes]
elif n_rows == 1 or n_cols == 1:
    axes = list(axes)
else:
    axes = axes.flatten()

for idx, word in enumerate(selfsim_target_words):
    ax = axes[idx]
    per_word = self_similarity_results_abs[idx]
    before_vals = [per_word[layer]["before"] for layer in layers]
    after_vals = [per_word[layer]["after"] for layer in layers]

    ax.plot(layers, before_vals, marker="o", linewidth=1.7, label="Before")
    ax.plot(layers, after_vals, marker="o", linewidth=1.7, label="After")
    ax.set_title(word)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Self-Sim")
    ax.grid(alpha=0.3)

# Hide any unused subplot slots
for idx in range(n_words, len(axes)):
    axes[idx].axis("off")

# Show one legend only to keep panels clean
if n_words > 0:
    axes[0].legend(loc="best")

fig.suptitle("Self-Similarity per Target Word (Before vs After Clipping)", fontsize=13)
plt.show()

## Experiment 7: Word sense

We load WiC sentence pairs and compute cosine similarity between contextualized vectors of the target word.

Prediction rule: if similarity >= threshold -> True label, else False label.

In [ ]:
from xai_bert import load_wic_splits, evaluate_wic_with_threshold

wic_train, wic_val, _ = load_wic_splits(
    seed=42,
 )

In [ ]:
for layer in range(1, 13):
	print(f"Layer {layer} dims to clip:", dims_to_clip_by_layer[layer])
	wic_eval = evaluate_wic_with_threshold(
		runner=runner,
		examples=wic_train,
		layer_idx=layer,
		threshold=0.70,
	)

	print("WiC eval:", wic_eval)

In [ ]:
dims_to_clip

In [ ]:
dims_to_clip_by_layer = {
    layer_idx: stats["top_min_dim"]
    for layer_idx, stats in outlier_stats.items()
}

In [ ]:
for layer in range(1, 13):
	print(f"Layer {layer} dims to clip:", dims_to_clip_by_layer[layer])
	wic_eval = evaluate_wic_with_threshold(
		runner=runner,
		examples=wic_train,
		layer_idx=layer,
		threshold=0.70,
		dims_to_clip=dims_to_clip_by_layer,
	)

	print("WiC eval:", wic_eval)

## Experiment 8: Pre-training Tiny BERT With vs Without Positional Embeddings

We run a short masked-language-model pretraining on WikiText-103 in two settings:
- standard positional embeddings
- positional embeddings disabled (zeroed and frozen)

Then we compare validation MLM loss in both settings.

In [5]:
wiki_train = load_wikitext_103_texts(split="train", max_texts=2000, seed=42)
wiki_val = load_wikitext_103_texts(split="validation", max_texts=400, seed=42)

In [6]:
runner_normal = BertHiddenStateRunner(model_name="bert-base-uncased", max_length=64)
runner_no_PE = BertHiddenStateRunner(model_name="bert-base-uncased", max_length=64)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
pretrain_with_pos = pretrain_tiny_bert_mlm(
    texts=wiki_train,
    model_name="bert-base-uncased",
    use_positional_embeddings=True,
    max_length=64,
    batch_size=32,
    num_steps=50,
    lr=5e-5,
)

pretrain_without_pos = pretrain_tiny_bert_mlm(
    texts=wiki_train,
    model_name="bert-base-uncased",
    use_positional_embeddings=False,
    max_length=64,
    batch_size=32,
    num_steps=50,
    lr=5e-5,
)

val_loss_with_pos = evaluate_mlm_loss(
    pretrain_with_pos["model"],
    pretrain_with_pos["tokenizer"],
    wiki_val,
    max_length=64,
    batch_size=32,
)
val_loss_without_pos = evaluate_mlm_loss(
    pretrain_without_pos["model"],
    pretrain_without_pos["tokenizer"],
    wiki_val,
    max_length=64,
    batch_size=32,
)

print("final train loss (with pos):", pretrain_with_pos["final_loss"])
print("final train loss (without pos):", pretrain_without_pos["final_loss"])
print("val MLM loss (with pos):", val_loss_with_pos)
print("val MLM loss (without pos):", val_loss_without_pos)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


final train loss (with pos): 2.1955621242523193
final train loss (without pos): 5.993154048919678
val MLM loss (with pos): 1.9492721832715547
val MLM loss (without pos): 5.8742085970365086


In [8]:
runner_normal.model = pretrain_with_pos["model"].to(runner_normal.device)
runner_normal.tokenizer = pretrain_with_pos["tokenizer"]
runner_normal.model.eval()

runner_no_PE.model = pretrain_without_pos["model"].to(runner_no_PE.device)
runner_no_PE.tokenizer = pretrain_without_pos["tokenizer"]
runner_no_PE.model.eval()

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [ ]:
mean_vectors = compute_mean_vectors_over_corpus(
    runner=runner_no_PE,
    texts=train_texts,
    batch_size=16,
    exclude_embedding_layer=True,
    remove_special_tokens=True,
)

plot_mean_vectors(mean_vectors)

## Experiment ? (New): Which Token Drives the Dominant Outlier? Maybe useless?

This experiment computes token-level responsibility for the dominant outlier dimension.

For each layer, we report:
- share of outlier events by token
- trigger rate of each token
- change in baseline outlier ratio if sentences containing that token are removed

In [ ]:
less_train_texts = train_texts[:1000]

In [ ]:
sample_text = ["Quick debug pass to inspect model output structure."]
inp = runner_normal.tokenizer(
    sample_text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=32,
)

device = next(runner_normal.model.parameters()).device
inp = {k: v.to(device) for k, v in inp.items()}

runner_normal.model.train()
out = runner_normal.model(
    **inp,
    output_hidden_states=True,
    output_attentions=True,
    return_dict=True,
)

print("out type:", type(out))
print("out keys:", list(out.keys()))

for k, v in out.items():
    if hasattr(v, "shape"):
        print(f"{k}: shape={tuple(v.shape)}")
    elif isinstance(v, (tuple, list)) and len(v) > 0 and hasattr(v[0], "shape"):
        print(f"{k}: len={len(v)}, first_shape={tuple(v[0].shape)}")
    else:
        print(f"{k}: {type(v)}")token_outlier_results = detect_token_outlier_responsibility(
    runner=runner,
    texts=less_train_texts,
    outlier_stats=outlier_stats,
    layers=(1, 6, 12),
    mode="min",
    batch_size=16,
    remove_special_tokens=False,
    min_occurrences=30,
    top_k=12,
)

print_token_outlier_summary(token_outlier_results, top_k=3)

For each layer, we show the top token.

In [ ]:
for layer, res in token_outlier_results.items():
	print(f"Layer {layer}:")
	print(f"token='{res['top_tokens'][0]['token']}'")